## Data Cleaning and Feature Engineering 

This notebook must produce the curated dataset:  `data/curated/part1_taxi_curated.parquet`

This curated dataset must contain the required geography, time, numeric, and engineered fields and we must apply the required cleaning rules.

#### Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

#### Defining file paths

In [2]:
tripdata_path = Path("../data/raw/tlc/yellow_tripdata_2023-01.parquet")
lookup_path = Path("../data/raw/lookup/taxi_zone_lookup.csv")
curated_path = Path("../data/curated/part1_taxi_curated.parquet")

#### Reloading datasets

In [3]:
tripdata = pd.read_parquet(tripdata_path)
zone_lookup = pd.read_csv(lookup_path)

In [5]:
tripdata.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [6]:
zone_lookup.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [7]:
print("Tripdata shape:", tripdata.shape)
print("Zone lookup shape:", zone_lookup.shape)

Tripdata shape: (3066766, 19)
Zone lookup shape: (265, 4)


#### Creating pickup and dropoff tables

In [8]:
pickup_lookup = zone_lookup.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "PU_borough",
    "Zone": "PU_zone"
})[["PULocationID", "PU_borough", "PU_zone"]]

In [9]:
dropoff_lookup = zone_lookup.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "DO_borough",
    "Zone": "DO_zone"
})[["DOLocationID", "DO_borough", "DO_zone"]]

#### Selecting only the required columns in the dataframe

In [10]:
df = tripdata[[
    "PULocationID",
    "DOLocationID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
]].copy()

#### Merging the zone lookup fields

In [11]:
df = df.merge(pickup_lookup, on="PULocationID", how="left")
df = df.merge(dropoff_lookup, on="DOLocationID", how="left")

In [12]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,PU_borough,PU_zone,DO_borough,DO_zone
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.30,0.00,14.30,2,Manhattan,Midtown Center,Manhattan,Lenox Hill West
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.90,4.00,16.90,1,Manhattan,Central Park,Manhattan,Upper East Side South
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.90,15.00,34.90,1,Manhattan,Clinton East,Manhattan,Upper West Side North
3,138,7,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,12.10,0.00,20.85,1,Queens,LaGuardia Airport,Queens,Astoria
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.40,3.28,19.68,1,Manhattan,Gramercy,Manhattan,East Village
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,107,48,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,15.80,3.96,23.76,0,Manhattan,Gramercy,Manhattan,Clinton East
3066762,112,75,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,22.43,2.64,29.07,0,Brooklyn,Greenpoint,Manhattan,East Harlem South
3066763,114,239,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,17.61,5.32,26.93,0,Manhattan,Greenwich Village South,Manhattan,Upper West Side South
3066764,230,79,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,18.15,4.43,26.58,0,Manhattan,Times Sq/Theatre District,Manhattan,East Village


Now the dataset includes:

- PU_borough, PU_zone
- DO_borough, DO_zone

which are required fields.

#### Converting timestamps for required time fields

In [13]:
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

#### Time fields

In [14]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df["weekday"] = pd.Categorical(
    df["tpep_pickup_datetime"].dt.strftime("%a"),
    categories=weekday_order,
    ordered=True
)

df["week_of_year"] = df["tpep_pickup_datetime"].dt.isocalendar().week.astype(int)

In [15]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,PU_borough,PU_zone,DO_borough,DO_zone,pickup_date,pickup_hour,weekday,week_of_year
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.30,0.00,14.30,2,Manhattan,Midtown Center,Manhattan,Lenox Hill West,2023-01-01,0,Sun,52
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.90,4.00,16.90,1,Manhattan,Central Park,Manhattan,Upper East Side South,2023-01-01,0,Sun,52
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.90,15.00,34.90,1,Manhattan,Clinton East,Manhattan,Upper West Side North,2023-01-01,0,Sun,52
3,138,7,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,12.10,0.00,20.85,1,Queens,LaGuardia Airport,Queens,Astoria,2023-01-01,0,Sun,52
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.40,3.28,19.68,1,Manhattan,Gramercy,Manhattan,East Village,2023-01-01,0,Sun,52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,107,48,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,15.80,3.96,23.76,0,Manhattan,Gramercy,Manhattan,Clinton East,2023-01-31,23,Tue,5
3066762,112,75,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,22.43,2.64,29.07,0,Brooklyn,Greenpoint,Manhattan,East Harlem South,2023-01-31,23,Tue,5
3066763,114,239,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,17.61,5.32,26.93,0,Manhattan,Greenwich Village South,Manhattan,Upper West Side South,2023-01-31,23,Tue,5
3066764,230,79,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,18.15,4.43,26.58,0,Manhattan,Times Sq/Theatre District,Manhattan,East Village,2023-01-31,23,Tue,5


This satisfies the required time fields:

- pickup_date
- pickup_hour
- weekday
- week_of_year

#### Engineered Fields

In [16]:
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [17]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,PU_borough,PU_zone,DO_borough,DO_zone,pickup_date,pickup_hour,weekday,week_of_year,trip_duration_min
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.30,0.00,14.30,2,Manhattan,Midtown Center,Manhattan,Lenox Hill West,2023-01-01,0,Sun,52,8.433333
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.90,4.00,16.90,1,Manhattan,Central Park,Manhattan,Upper East Side South,2023-01-01,0,Sun,52,6.316667
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.90,15.00,34.90,1,Manhattan,Clinton East,Manhattan,Upper West Side North,2023-01-01,0,Sun,52,12.750000
3,138,7,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,12.10,0.00,20.85,1,Queens,LaGuardia Airport,Queens,Astoria,2023-01-01,0,Sun,52,9.616667
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.40,3.28,19.68,1,Manhattan,Gramercy,Manhattan,East Village,2023-01-01,0,Sun,52,10.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,107,48,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,15.80,3.96,23.76,0,Manhattan,Gramercy,Manhattan,Clinton East,2023-01-31,23,Tue,5,13.983333
3066762,112,75,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,22.43,2.64,29.07,0,Brooklyn,Greenpoint,Manhattan,East Harlem South,2023-01-31,23,Tue,5,19.450000
3066763,114,239,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,17.61,5.32,26.93,0,Manhattan,Greenwich Village South,Manhattan,Upper West Side South,2023-01-31,23,Tue,5,24.516667
3066764,230,79,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,18.15,4.43,26.58,0,Manhattan,Times Sq/Theatre District,Manhattan,East Village,2023-01-31,23,Tue,5,13.000000


In [18]:
duration_hours = df["trip_duration_min"] / 60

df["speed_mph"] = np.where(
    duration_hours > 0,
    df["trip_distance"] / duration_hours,
    np.nan
)

df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    np.nan
)

df["tip_rate"] = np.where(
    df["total_amount"] > 0,
    df["tip_amount"] / df["total_amount"],
    np.nan
)

In [19]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,...,DO_borough,DO_zone,pickup_date,pickup_hour,weekday,week_of_year,trip_duration_min,speed_mph,fare_per_mile,tip_rate
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.30,0.00,14.30,2,...,Manhattan,Lenox Hill West,2023-01-01,0,Sun,52,8.433333,6.901186,9.587629,0.000000
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.90,4.00,16.90,1,...,Manhattan,Upper East Side South,2023-01-01,0,Sun,52,6.316667,10.448549,7.181818,0.236686
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.90,15.00,34.90,1,...,Manhattan,Upper West Side North,2023-01-01,0,Sun,52,12.750000,11.811765,5.936255,0.429799
3,138,7,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,12.10,0.00,20.85,1,...,Queens,Astoria,2023-01-01,0,Sun,52,9.616667,11.854419,6.368421,0.000000
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.40,3.28,19.68,1,...,Manhattan,East Village,2023-01-01,0,Sun,52,10.833333,7.920000,7.972028,0.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,107,48,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,15.80,3.96,23.76,0,...,Manhattan,Clinton East,2023-01-31,23,Tue,5,13.983333,13.087008,5.180328,0.166667
3066762,112,75,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,22.43,2.64,29.07,0,...,Manhattan,East Harlem South,2023-01-31,23,Tue,5,19.450000,17.892031,3.867241,0.090815
3066763,114,239,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,17.61,5.32,26.93,0,...,Manhattan,Upper West Side South,2023-01-31,23,Tue,5,24.516667,11.428960,3.770878,0.197549
3066764,230,79,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,18.15,4.43,26.58,0,...,Manhattan,East Village,2023-01-31,23,Tue,5,13.000000,14.538462,5.761905,0.166667


Distance Buckets: [0,1), [1,3), [3,5), [5,10), [10,+)

In [20]:
bins = [0, 1, 3, 5, 10, np.inf]
labels = ["[0,1)", "[1,3)", "[3,5)", "[5,10)", "[10,+)"]

df["distance_bucket"] = pd.cut(
    df["trip_distance"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

In [21]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,...,DO_zone,pickup_date,pickup_hour,weekday,week_of_year,trip_duration_min,speed_mph,fare_per_mile,tip_rate,distance_bucket
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.30,0.00,14.30,2,...,Lenox Hill West,2023-01-01,0,Sun,52,8.433333,6.901186,9.587629,0.000000,"[0,1)"
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.90,4.00,16.90,1,...,Upper East Side South,2023-01-01,0,Sun,52,6.316667,10.448549,7.181818,0.236686,"[1,3)"
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.90,15.00,34.90,1,...,Upper West Side North,2023-01-01,0,Sun,52,12.750000,11.811765,5.936255,0.429799,"[1,3)"
3,138,7,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,12.10,0.00,20.85,1,...,Astoria,2023-01-01,0,Sun,52,9.616667,11.854419,6.368421,0.000000,"[1,3)"
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.40,3.28,19.68,1,...,East Village,2023-01-01,0,Sun,52,10.833333,7.920000,7.972028,0.166667,"[1,3)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,107,48,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,15.80,3.96,23.76,0,...,Clinton East,2023-01-31,23,Tue,5,13.983333,13.087008,5.180328,0.166667,"[3,5)"
3066762,112,75,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,22.43,2.64,29.07,0,...,East Harlem South,2023-01-31,23,Tue,5,19.450000,17.892031,3.867241,0.090815,"[5,10)"
3066763,114,239,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,17.61,5.32,26.93,0,...,Upper West Side South,2023-01-31,23,Tue,5,24.516667,11.428960,3.770878,0.197549,"[3,5)"
3066764,230,79,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,18.15,4.43,26.58,0,...,East Village,2023-01-31,23,Tue,5,13.000000,14.538462,5.761905,0.166667,"[3,5)"


#### Pre-Cleaning copy of Dataset

In [22]:
df_before_clean = df.copy()
print("Rows before cleaning:", len(df_before_clean))

Rows before cleaning: 3066766


#### Applying Cleaning Rules

The assignment requires these exact rules:

1. Remove rows where pickup or dropoff timestamp is missing
2. Remove rows where `trip_duration_min <= 0`
3. Remove rows where `trip_distance <= 0`
4. Remove rows where `passenger_count` is missing or `<= 0`
5. Remove rows where `total_amount <= 0`
6. Define and remove duplicates using this key:
`(tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID, total_amount)`

And you must print a before/after table showing how many rows were removed by each rule.

In [23]:
summary_rows = []

def record_rule(rule_name, before_count, after_count):
    summary_rows.append({
        "rule": rule_name,
        "rows_before": before_count,
        "rows_after": after_count,
        "rows_removed": before_count - after_count
    })

In [24]:
# Rule 1: missing pickup or dropoff timestamp
before = len(df)
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
record_rule("Missing pickup/dropoff timestamp", before, len(df))

# Rule 2: trip_duration_min <= 0
before = len(df)
df = df[df["trip_duration_min"] > 0]
record_rule("trip_duration_min <= 0", before, len(df))

# Rule 3: trip_distance <= 0
before = len(df)
df = df[df["trip_distance"] > 0]
record_rule("trip_distance <= 0", before, len(df))

# Rule 4: passenger_count missing or <= 0
before = len(df)
df = df.dropna(subset=["passenger_count"])
df = df[df["passenger_count"] > 0]
record_rule("passenger_count missing or <= 0", before, len(df))

# Rule 5: total_amount <= 0
before = len(df)
df = df[df["total_amount"] > 0]
record_rule("total_amount <= 0", before, len(df))

# Rule 6: duplicates by required key
duplicate_key = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "total_amount"
]

before = len(df)
df = df.drop_duplicates(subset=duplicate_key)
record_rule("Duplicate rows by required key", before, len(df))

In [25]:
df

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,payment_type,...,DO_zone,pickup_date,pickup_hour,weekday,week_of_year,trip_duration_min,speed_mph,fare_per_mile,tip_rate,distance_bucket
0,161,141,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,9.3,0.00,14.30,2,...,Lenox Hill West,2023-01-01,0,Sun,52,8.433333,6.901186,9.587629,0.000000,"[0,1)"
1,43,237,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,7.9,4.00,16.90,1,...,Upper East Side South,2023-01-01,0,Sun,52,6.316667,10.448549,7.181818,0.236686,"[1,3)"
2,48,238,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,14.9,15.00,34.90,1,...,Upper West Side North,2023-01-01,0,Sun,52,12.750000,11.811765,5.936255,0.429799,"[1,3)"
4,107,79,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,11.4,3.28,19.68,1,...,East Village,2023-01-01,0,Sun,52,10.833333,7.920000,7.972028,0.166667,"[1,3)"
5,161,137,2023-01-01 00:50:34,2023-01-01 01:02:52,1.0,1.84,12.8,10.00,27.80,1,...,Kips Bay,2023-01-01,0,Sun,52,12.300000,8.975610,6.956522,0.359712,"[1,3)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995018,228,159,2023-01-31 23:00:19,2023-02-01 00:08:33,1.0,13.90,50.5,0.00,52.00,1,...,Melrose South,2023-01-31,23,Tue,5,68.233333,12.222765,3.633094,0.000000,"[10,+)"
2995019,263,107,2023-01-31 23:14:38,2023-01-31 23:25:30,1.0,3.37,15.6,2.00,22.60,1,...,Gramercy,2023-01-31,23,Tue,5,10.866667,18.607362,4.629080,0.088496,"[3,5)"
2995020,79,246,2023-01-31 23:44:51,2023-01-31 23:58:45,1.0,2.86,16.3,2.00,23.30,1,...,West Chelsea/Hudson Yards,2023-01-31,23,Tue,5,13.900000,12.345324,5.699301,0.085837,"[1,3)"
2995021,68,238,2023-01-31 23:45:00,2023-01-31 23:55:46,2.0,3.80,17.7,2.50,25.20,1,...,Upper West Side North,2023-01-31,23,Tue,5,10.766667,21.176471,4.657895,0.099206,"[3,5)"


In [ ]:
print("Rows after cleaning:", len(df))

Rows after cleaning: 2884396


: 

Cleaning Summary Table

In [26]:
cleaning_summary = pd.DataFrame(summary_rows)
cleaning_summary

,rule,rows_before,rows_after,rows_removed
0,Missing pickup/dropoff timestamp,3066766,3066766,0
1,trip_duration_min <= 0,3066766,3065645,1121
2,trip_distance <= 0,3065645,3020831,44814
3,passenger_count missing or <= 0,3020831,2906607,114224
4,total_amount <= 0,2906607,2884397,22210
5,Duplicate rows by required key,2884397,2884396,1


In [27]:
print("Final shape:", df.shape)
print("\nFinal columns:")
print(df.columns.tolist())

print("\nNull counts in curated dataset:")
print(df.isna().sum().sort_values(ascending=False).head(10))

Final shape: (2884396, 23)

Final columns:
['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'payment_type', 'PU_borough', 'PU_zone', 'DO_borough', 'DO_zone', 'pickup_date', 'pickup_hour', 'weekday', 'week_of_year', 'trip_duration_min', 'speed_mph', 'fare_per_mile', 'tip_rate', 'distance_bucket']

Null counts in curated dataset:
PU_zone                 37452
DO_zone                 18627
DO_borough               9420
PU_borough                476
PULocationID                0
DOLocationID                0
tpep_pickup_datetime        0
fare_amount                 0
trip_distance               0
passenger_count             0
dtype: int64


#### Saving curated dataset to:

`data/curated/part1_taxi_curated.parquet`

In [ ]:
curated_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(curated_path, index=False)

print(f"Saved curated dataset to: {curated_path}!")

Saved curated dataset to: ..\data\curated\part1_taxi_curated.parquet
